# Clustering

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import Datasets

In [ ]:
# Upload file jika menggunakan Google Colab
from google.colab import files
uploaded = files.upload()

# Membaca file Excel
wine_raw = pd.read_excel('wineQualityReds.xlsx', header=None)

# File yang diberikan tersimpan sebagai satu kolom berisi data CSV,
# sehingga dipisahkan berdasarkan tanda koma.
wine_columns = wine_raw.iloc[0, 0].split(',')
wine = wine_raw.iloc[1:, 0].astype(str).str.split(',', expand=True)
wine.columns = wine_columns

# Menghapus kolom pertama yang hanya berisi nomor baris/ID
wine = wine.drop(columns=wine.columns[0])

# Mengubah seluruh kolom menjadi tipe numerik
wine = wine.astype(float)
wine.info()

In [ ]:
wine.head()

## K-Means Clustering

In [ ]:
wine.head()

In [ ]:
# check missing values
wine.isna().sum()

In [ ]:
# check duplicates
wine.duplicated().sum()

### Data preprocessing

Seluruh kolom fitur numerik pada dataset digunakan untuk clustering. Kolom pertama yang berisi nomor baris dihapus karena hanya berfungsi sebagai identitas data, bukan karakteristik wine.

In [ ]:
# minmax scaler
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
wine_scaled = scaler.fit_transform(wine)

pd.DataFrame(wine_scaled, columns=wine.columns).describe()

In [ ]:
# elbow method from k=1 to k=10
from sklearn.cluster import KMeans

# track sum of squared error
sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(wine_scaled)
    sse.append(kmeans.inertia_)

In [ ]:
# draw the SSE decline progression
plt.figure(figsize=(12, 8))
plt.plot(range(1, 11), sse, marker='o')
plt.xticks(range(1, 11))
plt.xlabel('Number of Clusters')
plt.ylabel('Sum of Squared Error')
plt.show()

## Davies-Bouldin Index (DBI)

DBI digunakan untuk membantu menentukan jumlah cluster terbaik. Nilai DBI yang lebih kecil menunjukkan cluster yang relatif lebih kompak dan lebih terpisah.

In [ ]:
# calculate Davies-Bouldin Index for k=2 to k=10
from sklearn.metrics import davies_bouldin_score

dbi = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(wine_scaled)
    dbi.append(davies_bouldin_score(wine_scaled, labels))

# display DBI results
DBI_result = pd.DataFrame({
    'Number of Clusters (K)': range(2, 11),
    'Davies-Bouldin Index': dbi
})
DBI_result

In [ ]:
# choose the number of clusters with the lowest DBI
optimal_k = DBI_result.loc[DBI_result['Davies-Bouldin Index'].idxmin(), 'Number of Clusters (K)']
print('Optimal K berdasarkan DBI:', optimal_k)

Jumlah cluster terbaik dipilih berdasarkan nilai DBI paling rendah. Nilai ini juga dapat dibandingkan dengan bentuk grafik Elbow sebelum menentukan hasil akhir.

In [ ]:
# retrain k-means with optimal k
kmeans = KMeans(n_clusters=int(optimal_k), random_state=42, n_init=10)
kmeans.fit(wine_scaled)

# assign class label to wine df
wine['label'] = kmeans.labels_
wine.head()

In [ ]:
# compute mean of each feature column by group by label
wine.groupby('label').mean()

Cluster interpretation?

Bandingkan nilai rata-rata setiap karakteristik wine pada tiap label cluster. Perhatikan juga rata-rata `quality` untuk melihat bagaimana kualitas wine tersebar di masing-masing cluster. Ingat, clustering mengelompokkan data berdasarkan kemiripan fitur; hasilnya tidak otomatis membuktikan hubungan sebab-akibat.

# Selesai